# Laboratory 12 — Partition functions

In this laboratory you hand a computer the logarithm of a partition function — one number per
temperature — and ask it for the mean energy, the entropy, the free energy and the heat capacity.
It knows nothing else about the system: not the levels, not the probabilities, not the closed
forms. You then check every answer against the closed forms, and find out how far the numerical
derivatives can be trusted.

The route: the model and its objects (Part 1); a partition function by hand (Part 2); the
Z-machine on three systems, and the free energy tested against module 10's (Part 3); how the
difference step controls the error (Part 4); the paramagnet — Curie's law and its fluctuations
(Part 5); the paramagnet at fixed energy, negative temperatures and which way the heat goes
(Part 6); the harmonic oscillator, equipartition and freeze-out (Part 7); module 01's simulation
under the oscillator's partition function (Part 8); the automated checks (Part 9); and free
exploration (Part 10).

Work through it in order. Where the notebook asks you to predict, write your prediction in the
cell provided **before** running the next cell. A prediction you have committed to is the only
reliable way to discover that you were wrong.

## Model specification

| | |
|---|---|
| **System** | discrete-level systems with given levels: a two-level system (gap $\Delta$); $N$ independent moments $\mu$ in a field $B$, at $\mp\mu B$; a harmonic ladder $\hbar\omega(n + 1/2)$ |
| **Dynamics** | none: canonical averages from closed forms or explicit sums; only Part 8 inherits random numbers, from module 01's simulation |
| **Boundary** | contact with an infinite bath at $T$; the levels are fixed, so no work is done and every change of $U$ is heat |
| **Ensemble** | canonical, exactly: $P_s = e^{-\beta E_s}/Z$ |
| **Ignored** | interactions between subsystems, level shifts beyond the linear Zeeman term, identical-particle exchange |
| **Valid when** | the levels are fixed and known, and the subsystems are independent, so $Z_N = z^N$ |
| **Failure modes** | interacting moments (module 15); identical particles at high density or low $T$ (module 17); levels that move with the state variables |

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import equilibrium, partition
from thermolab.constants import K_B
from thermolab.validation import (
    convergence_study,
    relative_error,
    scaling_exponent,
    seed_study,
)

# partition.py draws no random numbers: it evaluates sums and closed forms. Part 8 runs module
# 01's simulation, which does, and every generator there is passed in explicitly.

QUANTUM = 1.0e-21  # J: the two-level gap and the oscillator's hbar*omega
MU = 9.274e-24  # J/T: one Bohr magneton, the moment of an electron spin
FIELD = 1.0  # T
print(f"quantum / k_B = {QUANTUM / K_B:.2f} K")
print(f"mu B / k_B    = {MU * FIELD / K_B:.4f} K")

## Part 1 — The objects

Every partition function in `thermolab.partition` is returned as its **logarithm**. $Z_N$ for ten
thousand moments is about $2^{10000}$ — no float can hold it — while $\ln Z_N$ is a modest number,
and every thermodynamic quantity needs only $\ln Z$.

| function | system | $\ln Z$ |
|---|---|---|
| `log_z_two_level(gap, T)` | levels $0$, $\Delta$ | $\ln(1 + e^{-\beta\Delta})$ |
| `log_z_paramagnet(N, mu, B, T)` | $N$ moments at $\mp\mu B$ | $N\ln(2\cosh\beta\mu B)$ |
| `log_z_harmonic(hbar_omega, T)` | ladder $\hbar\omega(n + 1/2)$ | $-x/2 - \ln(1 - e^{-x})$ |

And one machine, `thermo_from_z(log_z, temperatures)`, which is given only a function returning
$\ln Z$ and hands back $U$, $\sigma_E^2$, $C$, $F$ and $S$.

### Predict

Commit to an answer for each of the module page's four predictions before going further:

1. What does $\partial\ln Z/\partial\beta$ give, and with what sign?
2. Halve a paramagnet's temperature in a weak field. What happens to its magnetization?
3. Pour energy into the paramagnet past the point where half its moments point each way. What
   happens to its temperature — and which way does heat then flow into a room-temperature body?
4. How much energy does a hot harmonic oscillator hold, $\kB T \gg \hbar\omega$? A cold one?

**Your predictions:**

1.
2.
3.
4.

## Part 2 — A partition function by hand

Take the two-level system with gap $\Delta = 2\kB T$. On paper: write $Z$, then $\ln Z$, then
differentiate in $\beta$ to get $U$. Then run the next cell, which does the same three things —
by formula, and by a central difference of $\ln Z$ in $\beta$.

**Your $Z$ and $U$:**

*(write here before running the next cell)*

In [ ]:
temperature = QUANTUM / (2 * K_B)  # so that Delta = 2 k_B T
beta = 1 / (K_B * temperature)
z_by_hand = 1 + np.exp(-beta * QUANTUM)
print(f"Z by hand  = 1 + exp(-2)            = {z_by_hand:.6f}")
library_log_z = partition.log_z_two_level(QUANTUM, temperature)
print(f"ln Z       = {np.log(z_by_hand):.6f};  library: {library_log_z:.6f}")


def log_z_of_beta(b):
    return partition.log_z_two_level(QUANTUM, 1 / (K_B * b))


h = 1e-5 * beta
u_by_difference = -(log_z_of_beta(beta + h) - log_z_of_beta(beta - h)) / (2 * h)
u_by_formula = QUANTUM / (np.exp(2) + 1)
print(f"U = -d ln Z / d beta, by difference: {u_by_difference / QUANTUM:.8f} Delta")
print(f"U = Delta / (e^2 + 1), by formula:   {u_by_formula / QUANTUM:.8f} Delta")

Nothing in the difference knew what the levels were or how probable each one was. Differentiating
the logarithm of the normalization gave the mean energy.

## Part 3 — The Z-machine

`thermo_from_z` does the same for every quantity at once: central differences in $\beta$ for $U$
(first derivative) and for $\sigma_E^2$ (second derivative), then

$$
C = \frac{\sigma_E^2}{\kB T^2}, \qquad F = -\kB T\ln Z, \qquad S = \frac{U - F}{T} .
$$

Run it on all three systems, from a tenth of the level gap to ten times it, and plot the
reconstruction (dots) over the closed forms (lines).

In [ ]:
N_SPINS = 10_000
systems = {
    "two-level": dict(
        log_z=lambda t: partition.log_z_two_level(QUANTUM, t),
        energy=lambda t: partition.two_level_energy(QUANTUM, t),
        capacity=lambda t: partition.two_level_heat_capacity(QUANTUM, t),
        gap=QUANTUM, n=1),
    "paramagnet": dict(
        log_z=lambda t: partition.log_z_paramagnet(N_SPINS, MU, FIELD, t),
        energy=lambda t: partition.paramagnet_energy(N_SPINS, MU, FIELD, t),
        capacity=lambda t: partition.paramagnet_heat_capacity(N_SPINS, MU, FIELD, t),
        gap=2 * MU * FIELD, n=N_SPINS),
    "oscillator": dict(
        log_z=lambda t: partition.log_z_harmonic(QUANTUM, t),
        energy=lambda t: partition.harmonic_energy(QUANTUM, t),
        capacity=lambda t: partition.harmonic_heat_capacity(QUANTUM, t),
        gap=QUANTUM, n=1),
}

fig, axes = plt.subplots(3, 3, figsize=(12, 8.5), sharex="col")
residuals = {}
for column, (name, spec) in enumerate(systems.items()):
    x = np.geomspace(0.1, 10.0, 400)  # k_B T / gap
    t = x * spec["gap"] / K_B
    rebuilt = partition.thermo_from_z(spec["log_z"], t)
    middle = (x >= 0.2) & (x <= 5.0)
    residuals[name] = (
        np.max(np.abs(rebuilt.energy / spec["energy"](t) - 1)),
        np.max(np.abs(rebuilt.heat_capacity[middle] / spec["capacity"](t[middle]) - 1)),
    )
    per = spec["gap"] * spec["n"]
    show = slice(None, None, 20)
    axes[0, column].semilogx(x, spec["energy"](t) / per, "k")
    axes[0, column].semilogx(x[show], rebuilt.energy[show] / per, "o", color="#2563eb", ms=4)
    axes[1, column].semilogx(x, rebuilt.entropy / (K_B * spec["n"]), "k")
    axes[1, column].semilogx(x[show], rebuilt.entropy[show] / (K_B * spec["n"]), "o",
                             color="#2563eb", ms=4)
    axes[2, column].semilogx(x, spec["capacity"](t) / (K_B * spec["n"]), "k")
    axes[2, column].semilogx(x[show], rebuilt.heat_capacity[show] / (K_B * spec["n"]), "o",
                             color="#2563eb", ms=4)
    axes[0, column].set_title(name)
    axes[2, column].set_xlabel("k_B T / gap")
axes[0, 0].set_ylabel("U / (N gap)")
axes[1, 0].set_ylabel("S / (N k_B)")
axes[2, 0].set_ylabel("C / (N k_B)")
plt.tight_layout()
plt.show()

for name, (u_err, c_err) in residuals.items():
    print(f"{name:<11} largest relative error: U {u_err:.1e}   "
          f"C (a fifth to five gaps) {c_err:.1e}")

### Is $-\kB T\ln Z$ really module 10's free energy?

Module 10 defined $F = U - TS$ with no ensemble in sight. The claim $F = -\kB T\ln Z$ is a
theorem, and a theorem can be tested. Compute the oscillator's free energy **twice**, with
nothing shared between the two routes:

- from $\ln Z$: $F = -\kB T\ln Z$;
- from the probabilities: $U = \sum_n E_n P_n$ and the Gibbs entropy $S = -\kB\sum_n P_n\ln P_n$,
  then $F = U - TS$.

### Predict

Will they agree exactly, approximately, or only at high temperature?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
print("   T [K]   levels   -k_B T ln Z [J]          U - T S (Gibbs) [J]      relative gap")
free_energy_gaps = []
for temperature in (20.0, QUANTUM / K_B, 300.0):
    n_max = partition.harmonic_cutoff(QUANTUM, temperature)
    sums = partition.level_sums(QUANTUM * (np.arange(n_max + 1) + 0.5), temperature)
    gap = relative_error(sums.free_energy_direct, sums.free_energy_from_z)
    free_energy_gaps.append(gap)
    print(f"{temperature:8.2f}   {n_max + 1:6d}   {sums.free_energy_from_z:+.15e}   "
          f"{sums.free_energy_direct:+.15e}   {gap:.1e}")

# The falsifying experiment for "F = -k_B T ln Z is just a definition": two independent
# computations, one of them knowing nothing of Z, agree to the last digit a float can hold.
assert max(free_energy_gaps) < 1e-13

The two agree to rounding. $-\kB T\ln Z$ is not a new quantity that happens to share a name; it
*is* the $U - TS$ of module 10, with $S$ the entropy the probabilities carry.

## Part 4 — How small should the step be?

The central differences have a **truncation error** proportional to $h^2$: halve the step and
it quarters. But the formulas subtract values of $\ln Z$ that agree in more and more digits as
$h$ shrinks, and a float only has sixteen. That is **roundoff error**, and it grows as the step
shrinks — as $1/h$ for $U$, $1/h^2$ for $C$.

### Predict

As the relative step $h/\beta$ goes from $10^{-1}$ down to $10^{-10}$, sketch the error of $U$
and of $C$ on a log–log plot. Where is each smallest?

**Your sketch, in words:**

*(write here before running the next cell)*

In [ ]:
t_one = np.array([QUANTUM / K_B])  # k_B T = hbar omega
exact_u = partition.harmonic_energy(QUANTUM, t_one[0])
exact_c = partition.harmonic_heat_capacity(QUANTUM, t_one[0])
oscillator_log_z = systems["oscillator"]["log_z"]

relative_steps = np.geomspace(1e-1, 1e-10, 37)
u_errors, c_errors = [], []
for step in relative_steps:
    rebuilt = partition.thermo_from_z(oscillator_log_z, t_one, rel_step=step)
    u_errors.append(relative_error(rebuilt.energy[0], exact_u))
    c_errors.append(relative_error(rebuilt.heat_capacity[0], exact_c))

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.loglog(relative_steps, u_errors, "o-", color="#2563eb", ms=4, label="U")
ax.loglog(relative_steps, c_errors, "s-", color="#dc2626", ms=4, label="C")
ax.axvline(partition.FIRST_DERIVATIVE_REL_STEP, color="#2563eb", ls=":", lw=1)
ax.axvline(partition.SECOND_DERIVATIVE_REL_STEP, color="#dc2626", ls=":", lw=1)
ax.set_xlabel("h / beta")
ax.set_ylabel("relative error")
ax.invert_xaxis()
ax.legend()
plt.tight_layout()
plt.show()

print(f"default steps: {partition.FIRST_DERIVATIVE_REL_STEP:.1e} (U), "
      f"{partition.SECOND_DERIVATIVE_REL_STEP:.1e} (C)")
for label, errors in (("U", u_errors), ("C", c_errors)):
    best = relative_steps[np.argmin(errors)]
    print(f"smallest error of {label}: {min(errors):.1e} at h/beta = {best:.0e}")

The dotted lines are the defaults the machine uses: $\epsilon^{1/3}$ for $U$ and $\epsilon^{1/4}$
for $C$, with $\epsilon = 2.2\times10^{-16}$, the step sizes that balance the two errors. On the
right of each minimum the error falls with slope $2$; on the left it rises.

Now measure the order where truncation dominates, with an error bar — this is the laboratory's
measurement.

In [ ]:
refinements = [10, 20, 40, 80, 160]


def rebuilt_at(n):
    return partition.thermo_from_z(oscillator_log_z, t_one, rel_step=1.0 / n)


orders = {}
for label, attribute, exact in (("U", "energy", exact_u), ("C", "heat_capacity", exact_c)):
    study = convergence_study(lambda n, a=attribute: float(getattr(rebuilt_at(n), a)[0]),
                              refinements, exact)
    (slope, _), cov = np.polyfit(np.log(study.refinements), np.log(study.errors), 1, cov=True)
    orders[label] = (-slope, float(np.sqrt(cov[0, 0])))
    ratios = study.errors[:-1] / study.errors[1:]
    print(f"{label}: order p = {-slope:.4f} +/- {np.sqrt(cov[0, 0]):.4f}   "
          f"(each halving divides the error by {ratios.min():.3f} to {ratios.max():.3f})")

## Part 5 — The paramagnet

$N$ independent moments, each at $-\mu B$ along the field or $+\mu B$ against it. One moment has
$z = 2\cosh(\mu B/\kB T)$, and independence makes $\ln Z_N = N\ln z$ — no sum over $2^N$
microstates is ever taken. The magnetization is $M = N\mu\tanh(\mu B/\kB T)$.

### Predict

In a $1\ \mathrm{T}$ field, by what factor does $M$ change from $300\ \mathrm{K}$ to
$150\ \mathrm{K}$? From $1\ \mathrm{K}$ to $0.5\ \mathrm{K}$?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
additivity = (partition.log_z_paramagnet(4096, MU, FIELD, 2.0)
              / partition.log_z_paramagnet(1, MU, FIELD, 2.0))
print(f"additivity: ln Z(4096 moments) / ln Z(1 moment) = {additivity:.10f}")

pairs = [(300.0, 150.0), (1.0, 0.5), (0.2, 0.1)]
for hot, cold in pairs:
    ratio = (partition.paramagnet_magnetization(N_SPINS, MU, FIELD, cold)
             / partition.paramagnet_magnetization(N_SPINS, MU, FIELD, hot))
    print(f"M({cold} K) / M({hot} K) = {ratio:.5f}")

# Curie's law, fitted: in the weak-field regime M is proportional to 1/T; the slope is the
# Curie constant N mu^2 B / k_B.
curie_t = np.linspace(50.0, 300.0, 11)
curie_m = partition.paramagnet_magnetization(N_SPINS, MU, FIELD, curie_t)
(curie_slope, curie_intercept), curie_cov = np.polyfit(1 / curie_t, curie_m, 1, cov=True)
curie_expected = N_SPINS * MU**2 * FIELD / K_B
print(f"\nfitted Curie constant: {curie_slope:.5e} +/- {np.sqrt(curie_cov[0, 0]):.1e} J K/T")
print(f"N mu^2 B / k_B:        {curie_expected:.5e} J K/T")

x = np.geomspace(0.01, 10, 300)  # mu B / (k_B T)
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
left.semilogx(x, np.tanh(x), "k", label="N mu tanh(mu B / k_B T)")
left.semilogx(x, x, "--", color="#2563eb", label="Curie law")
left.axhline(1.0, color="grey", ls=":")
left.set_ylim(0, 1.2)
left.set_xlabel("mu B / (k_B T)")
left.set_ylabel("M / (N mu)")
left.legend()
right.plot(1 / curie_t, curie_m / (N_SPINS * MU), "o", color="#2563eb")
right.plot(1 / curie_t, (curie_intercept + curie_slope / curie_t) / (N_SPINS * MU), "k-")
right.set_xlabel("1 / T  [1/K]")
right.set_ylabel("M / (N mu)")
right.set_title("Curie regime: M against 1/T")
plt.tight_layout()
plt.show()

### The energy fluctuations

The second derivative of $\ln Z_N$ is the variance of the energy. Independence makes both $U$
and $\sigma_E^2$ grow as $N$, so $\sigma_E/|U|$ should fall as $N^{-1/2}$. Measure the exponent
from the machine's own second derivatives — no closed form involved.

In [ ]:
sizes = [10, 100, 1_000, 10_000, 100_000, 1_000_000]
relative_spread = []
for n in sizes:
    rebuilt = partition.thermo_from_z(
        lambda t, n=n: partition.log_z_paramagnet(n, MU, FIELD, t), np.array([1.0]))
    relative_spread.append(float(np.sqrt(rebuilt.energy_variance[0]) / abs(rebuilt.energy[0])))
    print(f"N = {n:>9,d}   sigma_E / |U| = {relative_spread[-1]:.6f}")
fluctuation_exponent = scaling_exponent(sizes, relative_spread)
print(f"\nfitted exponent: {fluctuation_exponent:.7f}  (the derivation says -1/2)")

## Part 6 — The paramagnet at fixed energy, and negative temperature

Isolate the paramagnet. With $n$ of its $N$ moments reversed, $U = \mu B(2n - N)$ and
$S = \kB\ln\binom{N}{n}$ — module 08's coins again. The temperature is module 09's slope,
$1/T = \partial S/\partial U$.

### Predict

Sketch $S(U)$ from $U = -N\mu B$ to $+N\mu B$. Where is its slope zero? What is the sign of $T$
past that point?

**Your sketch, in words:**

*(write here before running the next cell)*

In [ ]:
spins = partition.spin_entropy_of_energy(100, MU, FIELD)
beta_dome = spins.beta()
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
left.plot(spins.energy / (MU * FIELD), spins.entropy / K_B, "k")
left.set_xlabel("U / (mu B)")
left.set_ylabel("S / k_B")
right.plot(spins.energy / (MU * FIELD), beta_dome * MU * FIELD, "o", color="#2563eb", ms=3)
right.axhline(0, color="grey", lw=1)
right.set_xlabel("U / (mu B)")
right.set_ylabel("beta mu B")
plt.tight_layout()
plt.show()

print("reversed   U/(mu B)   S/k_B     beta mu B    T [K]")
for n in (10, 30, 49, 50, 51, 70, 85):
    b = beta_dome[n]
    t_text = "  +/-inf" if b == 0 else f"{1 / (K_B * b):+8.3f}"
    print(f"{n:8d}   {spins.energy[n] / (MU * FIELD):+8.0f}   {spins.entropy[n] / K_B:7.3f}"
          f"   {b * MU * FIELD:+9.4f}   {t_text}")

### Which way does the heat go?

Prepare the paramagnet with $85$ of its $100$ moments reversed: $T \approx -0.79\ \mathrm{K}$.
Touch it to an Einstein solid of $200$ oscillators whose quantum is one reversal, $2\mu B$, and
count every way the pair can share its energy. Try the solid cold, warm and very hot.

### Predict

For each solid temperature, does energy flow into the paramagnet or out of it?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
N_OSC = 200
flip = 2 * MU * FIELD
print(" solid quanta/osc   solid T [K]   reversed: start -> most probable   quanta to solid"
      "   dlnOmega(one quantum out)   (one quantum in)")
contacts = {}
for per_oscillator in (0.1, 1.0, 10.0, 1000.0):
    contact = partition.spin_solid_contact(100, 85, N_OSC, int(per_oscillator * N_OSC))
    contacts[per_oscillator] = contact
    solid_t = partition.harmonic_temperature(per_oscillator * flip, flip)
    print(f"{per_oscillator:16.1f}   {solid_t:11.2f}   {contact.initial_flipped:8d} -> "
          f"{contact.most_probable_flipped:3d}              {contact.energy_to_solid:5d}"
          f"            {contact.log_omega_change(-1):+.3f}                   "
          f"{contact.log_omega_change(+1):+.3f}")

# The falsifying experiment for "negative temperature is colder than absolute zero": at every
# solid temperature, total multiplicity rises only when the inverted spins LOSE energy.
for contact in contacts.values():
    assert contact.energy_to_solid > 0
    assert contact.log_omega_change(-1) > 0 > contact.log_omega_change(+1)

Energy leaves the paramagnet every time — even for the solid at over a thousand kelvin. A
negative temperature is hotter than any positive one: ordered by $-\beta$, it sits beyond
$T = +\infty$, not below $T = 0$.

## Part 7 — The harmonic oscillator

The ladder $\hbar\omega(n + 1/2)$ sums to a geometric series, $z = e^{-x/2}/(1 - e^{-x})$ with
$x = \hbar\omega/\kB T$. First check the closed form against the explicit sum over levels, cut off
where the Boltzmann factors have died away. Then look at both ends of the temperature range.

In [ ]:
print("  T [K]    n_max    ln z (sum)            ln z (closed form)")
for temperature in (5.0, 72.43, 2000.0):
    n_max = partition.harmonic_cutoff(QUANTUM, temperature)
    summed = partition.log_z_harmonic(QUANTUM, temperature, n_max=n_max)
    closed = partition.log_z_harmonic(QUANTUM, temperature)
    print(f"{temperature:7.2f}   {n_max:6d}   {summed:.15f}   {closed:.15f}")

x = np.geomspace(0.05, 50, 400)  # k_B T / (hbar omega)
t = x * QUANTUM / K_B
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
left.loglog(x, partition.harmonic_energy(QUANTUM, t, zero_point=False) / QUANTUM, "k",
            label="U above the ground state")
left.loglog(x, x, "--", color="#2563eb", label="equipartition, k_B T")
left.set_xlabel("k_B T / (hbar omega)")
left.set_ylabel("U / (hbar omega)")
left.legend()
right.semilogx(x, partition.harmonic_heat_capacity(QUANTUM, t) / K_B, "k")
right.axhline(1.0, color="#2563eb", ls="--")
right.set_xlabel("k_B T / (hbar omega)")
right.set_ylabel("C / k_B")
plt.tight_layout()
plt.show()

for ratio in (100.0, 10.0, 1.0, 0.2, 0.1):
    temperature = ratio * QUANTUM / K_B
    above_ground = partition.harmonic_energy(QUANTUM, temperature, zero_point=False)
    capacity = partition.harmonic_heat_capacity(QUANTUM, temperature)
    print(f"k_B T = {ratio:5.1f} hbar omega:  U/k_B T (above ground) = "
          f"{above_ground / (K_B * temperature):.4f}   C/k_B = {capacity / K_B:.4f}")

Hot, the oscillator holds $\kB T$ — equipartition, the rule modules 04 and 06 assumed, now a
limit. Cold, its heat capacity dies exponentially: it can only take energy a whole quantum at a
time.

## Part 8 — Module 01's solid, re-read

Module 01 simulated two Einstein solids exchanging quanta and read each one's temperature by
equipartition, $T = U/(n\kB)$. Its model specification warned that this fails when cold. The
oscillator partition function gives the exact reading, $T = \hbar\omega/[\kB\ln(1 + n\hbar\omega/U)]$.

Run module 01's simulation at several energies, with eight seeds each, and read the endpoints
both ways.

### Predict

At $0.1$ quanta per oscillator, will module 01's reading be too hot, too cold, or right?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
N_BODY = 40


def endpoint_temperature(rng, state):
    # Body A's temperature, read through Z, averaged over the second half of the run.
    run = equilibrium.simulate_energy_exchange(state, 20 * state.total_quanta, rng)
    tail = run.q_a[len(run.q_a) // 2:]
    return float(partition.harmonic_temperature(tail.mean() / N_BODY * QUANTUM, QUANTUM))


print("quanta/osc   module 01 reads [K]   Z predicts [K]   simulated, read by Z [K]")
overlay = []
for per_oscillator in (0.1, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0):
    t_equipartition = per_oscillator * QUANTUM / K_B
    state = equilibrium.from_temperatures(N_BODY, N_BODY, 1.6 * t_equipartition,
                                          0.4 * t_equipartition, QUANTUM)
    study = seed_study(lambda rng, s=state: endpoint_temperature(rng, s), n_seeds=8)
    shared = state.total_quanta / state.total_oscillators
    predicted = partition.harmonic_temperature(shared * QUANTUM, QUANTUM)
    overlay.append((shared, predicted, study))
    print(f"{shared:9.3f}   {shared * QUANTUM / K_B:18.1f}   {predicted:14.1f}   "
          f"{study.mean:10.1f} +/- {study.standard_error:.1f}")

shared_u = np.array([row[0] for row in overlay])
fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.geomspace(0.05, 20, 300)
ax.loglog(x, 1 / np.expm1(1 / x), "k", label="oscillator Z")
ax.loglog(x, x, "--", color="grey", label="equipartition (module 01)")
ax.errorbar([row[2].mean * K_B / QUANTUM for row in overlay], shared_u,
            xerr=[row[2].standard_error * K_B / QUANTUM for row in overlay],
            fmt="o", color="#2563eb", label="simulated endpoints, read by Z")
ax.plot(shared_u, shared_u, "o", mfc="none", color="grey",
        label="same endpoints, read by module 01")
ax.set_xlabel("k_B T / (hbar omega)")
ax.set_ylabel("quanta per oscillator")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

The simulation supplies energies; the partition function supplies the thermometer. Hot, the two
readings differ by half a quantum. At $0.1$ quanta per oscillator module 01's reading is four
times too cold — exactly the regime its model specification excluded. The overlay illustrates;
step 7 of the derivation establishes.

## Part 9 — Automated checks

A calculation you have not checked is a picture, not evidence. These are the same assertions
that run in the project's test suite.

In [ ]:
# 1. By hand and by difference, U of the two-level system at Delta = 2 k_B T (Part 2).
assert relative_error(u_by_difference, u_by_formula) < 1e-8

# 2. The Z-machine reproduces every closed form (Part 3) ...
for u_err, c_err in residuals.values():
    assert u_err < 1e-6 and c_err < 1e-4
# ... and -k_B T ln Z is U - TS with the Gibbs entropy (checked above).

# 3. The stencil is second order (Part 4).
for order, error in orders.values():
    assert abs(order - 2.0) < 5 * error + 0.01

# 4. Curie's law: halving T in a weak field doubles M; the fitted constant is N mu^2 B / k_B.
assert relative_error(curie_slope, curie_expected) < 1e-3
# 5. Relative fluctuations fall as N^(-1/2) (Part 5).
assert abs(fluctuation_exponent + 0.5) < 1e-4

# 6. The dome's slope changes sign at the top, and inverted spins give energy away (Part 6).
assert beta_dome[50] == 0 and beta_dome[49] > 0 > beta_dome[51]

# 7. Every simulated endpoint agrees with Z's temperature, within three standard errors (Part 8).
for _, predicted, study in overlay:
    assert study.agrees_with(predicted)

print("all checks passed")

## Part 10 — Explore it yourself

Choose a system and its level scale in units of a reference temperature $T_0 = 100\ \mathrm{K}$,
then press **Run Interact**. The panels show $U$, $S$, $F$ and $C$ reconstructed from $\ln Z$
(dots) over the exact curves (lines), and the largest relative error of each. Three things worth
trying:

1. Make the oscillator's quantum large. Where on the temperature axis does $C$ switch on?
2. Give the paramagnet a single moment, then ten thousand. What changes in the plots, and what
   does not, once each quantity is divided by $N$?
3. Push the level scale to the edge of the slider. In which panel does the reconstruction first
   show a visible error, and why that one?

In [ ]:
import ipywidgets as widgets

T_REFERENCE = 100.0  # K


def explore(system="oscillator", scale=1.0, log10_spins=4.0):
    level = scale * K_B * T_REFERENCE
    n = int(round(10**log10_spins))
    if system == "two-level":
        log_z = lambda t: partition.log_z_two_level(level, t)  # noqa: E731
        exact = (partition.two_level_energy(level, temps), None,
                 partition.two_level_heat_capacity(level, temps))
        n = 1
    elif system == "paramagnet":
        moment = level / FIELD
        log_z = lambda t: partition.log_z_paramagnet(n, moment, FIELD, t)  # noqa: E731
        exact = (partition.paramagnet_energy(n, moment, FIELD, temps),
                 partition.paramagnet_entropy(n, moment, FIELD, temps),
                 partition.paramagnet_heat_capacity(n, moment, FIELD, temps))
    else:
        log_z = lambda t: partition.log_z_harmonic(level, t)  # noqa: E731
        exact = (partition.harmonic_energy(level, temps), partition.harmonic_entropy(level, temps),
                 partition.harmonic_heat_capacity(level, temps))
        n = 1
    rebuilt = partition.thermo_from_z(log_z, temps)
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
    series = (("U / (N k_B T0)", rebuilt.energy, exact[0], K_B * T_REFERENCE),
              ("S / (N k_B)", rebuilt.entropy, exact[1], K_B),
              ("F / (N k_B T0)", rebuilt.free_energy, None, K_B * T_REFERENCE),
              ("C / (N k_B)", rebuilt.heat_capacity, exact[2], K_B))
    for ax, (label, value, reference, unit) in zip(axes, series, strict=True):
        if reference is not None:
            ax.semilogx(temps, reference / (n * unit), "k")
            worst = np.max(np.abs(value / reference - 1))
            ax.set_title(f"max rel. error {worst:.0e}", fontsize=9)
        ax.semilogx(temps[::15], value[::15] / (n * unit), "o", color="#2563eb", ms=3)
        ax.set_xlabel("T [K]")
        ax.set_ylabel(label)
    plt.tight_layout()
    plt.show()


temps = np.geomspace(1.0, 10_000.0, 300)
widgets.interact_manual(
    explore,
    system=["oscillator", "two-level", "paramagnet"],
    scale=widgets.FloatLogSlider(value=1.0, base=10, min=-1, max=1, step=0.05),
    log10_spins=widgets.FloatSlider(value=4.0, min=0.0, max=4.0, step=0.5),
)

## Check your understanding

Run the cell below for the auto-graded quiz. The same questions, with written explanations
for every option, are on the module page.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "12-partition-functions.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## Before you leave

Write a few sentences on each, in the cell below.

1. What did you predict that turned out to be wrong, and what specifically was the flaw in
   your reasoning?
2. The Z-machine never saw a level or a probability. Explain, without equations, how it found
   the heat capacity.
3. In Part 4 a smaller step made the answer *worse*. Why, and why did $C$ suffer before $U$?
4. In Part 6 a paramagnet at $-0.79\ \mathrm{K}$ heated a solid at $1344\ \mathrm{K}$. Explain
   this to someone who believes negative temperatures are below absolute zero.

**Your answers:**

1.
2.
3.
4.